In [2]:
# from core.XGBoostModel import XGBoostModel
# from core.RunData import RunPipeline
import pandas as pd
import optuna
from sklearn.preprocessing import LabelEncoder
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold ,train_test_split, cross_val_score
import kagglehub
import os
import zipfile
# from core.seed_utils import SEED
import requests


# GenericDataPipeline

In [2]:
class GenericDataPipeline:

    def rank_features(self, X, y, n_folds=5):
        print(f"Starting feature importance calculation with XGBoost and {n_folds}-fold CV...")

        X_copy = X.copy()

        null_ratios = {}
        for col in X_copy.columns:
            null_ratios[col] = X_copy[col].isna().mean()

        for col in X_copy.select_dtypes(include=['int64', 'float64']).columns:
            X_copy[col] = X_copy[col].replace([np.inf, -np.inf], np.nan)
            
            non_null = X_copy[col].dropna()
            if len(non_null) > 0:
                mean_val = non_null.mean()
                std_val = non_null.std()
                if std_val > 0 and not np.isnan(mean_val):
                    upper_bound = mean_val + 5 * std_val
                    lower_bound = mean_val - 5 * std_val
                    X_copy[col] = X_copy[col].clip(lower=lower_bound, upper=upper_bound)
        
        num_cols = X_copy.select_dtypes(include=['int64', 'float64']).columns
        cat_cols = X_copy.select_dtypes(include=['object']).columns
        
        X_processed = X_copy.copy()
        
        for col in cat_cols:
            if X_processed[col].isna().sum() > 0:
                most_freq = X_processed[col].mode()[0] if not X_processed[col].isna().all() else 'Unknown'
                X_processed[col] = X_processed[col].fillna(most_freq)
            X_processed[col] = X_processed[col].astype('category')
        
        for col in num_cols:
            if X_processed[col].isna().sum() > 0:
                median_val = X_processed[col].median() if not X_processed[col].isna().all() else 0
                X_processed[col] = X_processed[col].fillna(median_val)
        
        X_filled = X_processed

        mean_target = float(np.mean(y))
        print(f"Mean target value (for base_score): {mean_target:.6f}")

        base_score = 0.5
        if 0 < mean_target < 1:
            base_score = mean_target

        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'auc',
            'max_depth': 6,
            'eta': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_weight': 3,
            'alpha': 0.01,
            'lambda': 1.0,
            'gamma': 0.1,
            'base_score': base_score, 
            'seed': 42,
            'tree_method': 'hist',
            'device': 'cuda'

        }

        kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

        fold_importances = []

        fold_scores = []

        print(f"Training XGBoost models across {n_folds} folds...")

        y_values = set(y.unique())
        print(f"Unique target values: {y_values}")

        if not all(isinstance(val, (int, float)) for val in y_values):
            print("Warning: Target variable contains non-numeric values. Converting to numeric.")
            y = y.astype(float)

        if not all(val in [0, 1] for val in y_values):
            print("Warning: Target variable contains values other than 0 and 1.")
            print("Converting target to binary: 0 for negative class, 1 for positive class.")
            y = (y > 0).astype(int)

        # Perform cross-validation
        for fold, (train_idx, val_idx) in enumerate(kf.split(X_filled, y)):
            print(f"Fold {fold+1}/{n_folds}")

            X_train, X_val = X_filled.iloc[train_idx], X_filled.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            train_pos = np.mean(y_train)
            val_pos = np.mean(y_val)
            print(f"  Train positive rate: {train_pos:.4f}, Validation positive rate: {val_pos:.4f}")

            try:
                dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
                dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

                # Train model
                model = xgb.train(
                    params,
                    dtrain,
                    num_boost_round=100,
                    early_stopping_rounds=20,
                    evals=[(dval, 'validation')],
                    verbose_eval=False
                )

                y_pred = model.predict(dval)
                score = roc_auc_score(y_val, y_pred)
                fold_scores.append(score)

                try:
                    importance = model.get_score(importance_type='gain')

                    fold_importance = {col: importance.get(col, 0) for col in X_filled.columns}

                    max_value = max(fold_importance.values()) if max(fold_importance.values()) > 0 else 1
                    normalized_importance = {col: val/max_value for col, val in fold_importance.items()}

                    fold_importances.append(normalized_importance)
                except Exception as e:
                    print(f"Warning: Could not get feature importance for fold {fold+1}: {str(e)}")

            except Exception as e:
                print(f"Error in fold {fold+1}: {str(e)}")
                print("Skipping this fold and continuing with others.")

        if fold_scores:
            mean_auc = np.mean(fold_scores)
            print(f"Mean AUC across {len(fold_scores)} successful folds: {mean_auc:.4f}")
        else:
            print("No successful folds. Cannot calculate mean AUC.")

        if not fold_importances:
            print("No feature importance information could be gathered from any fold.")
            print("Using a simple fallback method for feature scoring.")

            avg_importance = {}
            for col in X_filled.columns:
                try:
                    valid_mask = ~pd.isna(X_copy[col]) & ~pd.isna(y)
                    if valid_mask.sum() > 10: 
                        if X_filled[col].dtype.name == 'category':
                            col_values = X_filled[col][valid_mask].cat.codes
                        else:
                            col_values = X_filled[col][valid_mask]
                        
                        y_values = y[valid_mask]
                        corr = abs(np.corrcoef(col_values, y_values)[0, 1])
                        avg_importance[col] = corr if not np.isnan(corr) else 0
                    else:
                        avg_importance[col] = np.random.uniform(0, 0.001)
                except Exception as e:
                    print(f"Warning: Could not calculate correlation for {col}: {str(e)}")
                    avg_importance[col] = np.random.uniform(0, 0.001)
        else:
            avg_importance = {}
            for col in X_filled.columns:
                importances = [fold_imp.get(col, 0) for fold_imp in fold_importances]
                if importances:
                    avg_importance[col] = np.mean(importances)
                else:
                    null_ratio = null_ratios.get(col, 0)
                    avg_importance[col] = max(0.001 * (1 - null_ratio), 0.0001)  

        feature_data = []
        for col in X_copy.columns:
            gain_score = avg_importance.get(col, 0)
            null_ratio = null_ratios.get(col, 0)
            feature_data.append({
                'feature_name': col,
                'gain_score': gain_score,
                'null_ratio': null_ratio
            })
        
        feature_df = pd.DataFrame(feature_data)
        feature_df = feature_df.sort_values('gain_score', ascending=False)
        
        print("\nFeature scores:")
        print("-" * 100)
        print(f"{'Rank':<5} | {'Feature':<30} | {'Gain Score':<15} | {'Null Ratio':<10}")
        print("-" * 100)

        for i, row in feature_df.iterrows():
            rank = i + 1
            feat = row['feature_name']
            gain = row['gain_score']
            null_ratio = row['null_ratio']
            print(f"{rank:<5} | {feat:<30} | {gain:.6f} | {null_ratio:.4f}")

        return feature_df

    def objective(self, trial, feature_scores_with_nulls, all_features_score, df, label="readmitted"):
        K = 10
        selected_features = feature_scores_with_nulls['feature_name'].to_list()[:K]
        print("selected_features")
        print(selected_features)
        all_features = all_features_score['feature_name'].to_list()
        # Create binary assignment for each of the selected features:
        assignment = {}
        for feat in selected_features:
            # 1 means feature goes to stage1, 0 means feature is used in stage2
            assignment[feat] = trial.suggest_categorical(f"assign_{feat}", [1, 0])
        
        for feat in all_features:
            if feat not in assignment.keys():
                assignment[feat] = 1
        vals = 0
        for key,value in assignment.items():
            vals +=value
        if vals==0:
            return 99999
        penalty = 0 
        # Derive feature sets based on the assignment:
        stage1_features = [feat for feat, assign in assignment.items() if assign == 1]
        stage2_features = [feat for feat, assign in assignment.items() if assign == 0]
        csv_name = f"trial_{trial.number}_results.csv"
        dm = RunPipeline()
        validation_loss = dm.full_run(df,stage1_features,stage2_features,label,csv_name)
        if validation_loss == 999:
            return validation_loss
        else:
            final_objective = validation_loss + penalty
            return final_objective
    def preprocessing(self,df):
        for col in df.columns:
            if df[col].dtype == 'object':
                # Replace ? with NaN
                df[col] = df[col].replace(['?', ''], np.nan)
                # Convert to categorical codes
                if df[col].isna().sum() < len(df):  # If not all values are NaN
                    df[col] = pd.Categorical(df[col]).codes

                    # Ensure -1 (missing) values are converted to NaN
                    df[col] = df[col].replace(-1, np.nan)
        # Convert boolean columns to int
        for col in df.select_dtypes(include=['bool']).columns:
            df[col] = df[col].astype(int)
        return df 
    def diabetic_data(self,file_path,n_trials=20):
        
        print(f"Loading data from {file_path}...")
        df = pd.read_csv(file_path)
        df['readmitted'] = df['readmitted'].replace({'NO': 0, '<30': 1, '>30': 1}).astype(int)
        print("Distribution after mapping:")
        print(df['readmitted'].value_counts())
        print(df['readmitted'].dtype)
        
        print("Preprocessing data...")
        df = self.preprocessing(df)
        if 'number_inpatient' in df.columns:
            print("Dropping 'number_inpatient' column as requested")
            df = df.drop(['number_inpatient','number_emergency','discharge_disposition_id'], axis=1)
        return df



# diabetic_data

In [6]:
fs = GenericDataPipeline()
file_path = "/sise/home/daniel7/IncrementalLearing/Changes /diabetic_data.csv"
study = fs.diabetic_data(file_path,20)


Loading data from /sise/home/daniel7/IncrementalLearing/Changes /diabetic_data.csv...


/tmp/ipykernel_2644745/1326248478.py:473: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['readmitted'] = df['readmitted'].replace({'NO': 0, '<30': 1, '>30': 1}).astype(int)


Distribution after mapping:
readmitted
0    54864
1    46902
Name: count, dtype: int64
int64
Preprocessing data...
Dropping 'number_inpatient' column as requested
Using 'readmitted' as target variable
Starting feature importance calculation with XGBoost and 5-fold CV...
Mean target value (for base_score): 0.460881
Training XGBoost models across 5 folds...
Unique target values: {np.int64(0), np.int64(1)}
Fold 1/5
  Train positive rate: 0.4609, Validation positive rate: 0.4609
Fold 2/5
  Train positive rate: 0.4609, Validation positive rate: 0.4609
Fold 3/5
  Train positive rate: 0.4609, Validation positive rate: 0.4609
Fold 4/5
  Train positive rate: 0.4609, Validation positive rate: 0.4609
Fold 5/5
  Train positive rate: 0.4609, Validation positive rate: 0.4609


[I 2025-05-17 12:59:03,205] A new study created in memory with name: no-name-81069196-7b98-470b-832c-8a3e08565a73


Mean AUC across 5 successful folds: 0.6970

Feature scores:
----------------------------------------------------------------------------------------------------
Rank  | Feature                        | Gain Score      | Null Ratio
----------------------------------------------------------------------------------------------------
15    | number_outpatient              | 1.000000 | 0.0000
46    | diabetesMed                    | 0.480657 | 0.0000
19    | number_diagnoses               | 0.464990 | 0.0000
2     | patient_nbr                    | 0.414139 | 0.0000
8     | admission_source_id            | 0.387119 | 0.0000
1     | encounter_id                   | 0.366691 | 0.0000
14    | num_medications                | 0.248206 | 0.0000
16    | diag_1                         | 0.232529 | 0.0002
13    | num_procedures                 | 0.232094 | 0.0000
10    | payer_code                     | 0.231637 | 0.3956
22    | metformin                      | 0.220619 | 0.0000
5     | age        

...

********** run results **********
{'max_depth': 4, 'learning_rate': 0.014166466374276925, 'n_estimators': 679, 'min_child_weight': 5, 'subsample': 0.8160353403108233, 'colsample_bytree': 0.6037707441689445, 'gamma': 3.9429179901550295, 'lambda': 9.777465639392062, 'alpha': 0.8367459900757892, 'scale_pos_weight': 2.7870844103373837, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.6848104910122963), np.float64(0.6886370228443932), np.float64(0.6886138953763037)]
********** run results **********


/sise/home/daniel7/IncrementalLearing/Changes /XGBoostModel.py:21: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
[I 2025-05-17 14:16:18,100] Trial 19 finished with value: 0.6850447952055697 and parameters: {'max_depth': 6, 'learning_rate': 0.004499540495798365, 'n_estimators': 886, 'min_child_weight': 3, 'subsample': 0.9013700121848707, 'colsample_bytree': 0.8869745841042808, 'gamma': 2.5388965059149213, 'lambda': 2.919185939942285, 'alpha': 2.1213304098023626, 'scale_pos_weight': 6.290343720284795}. Best is trial 11 with value: 0.6407981998668543.


********** run results **********
{'max_depth': 6, 'learning_rate': 0.004499540495798365, 'n_estimators': 886, 'min_child_weight': 3, 'subsample': 0.9013700121848707, 'colsample_bytree': 0.8869745841042808, 'gamma': 2.5388965059149213, 'lambda': 2.919185939942285, 'alpha': 2.1213304098023626, 'scale_pos_weight': 6.290343720284795, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.6823481995901571), np.float64(0.6866319075608633), np.float64(0.6861542784656885)]
********** run results **********
Best parameters found: {'max_depth': 3, 'learning_rate': 0.0010114028296617254, 'n_estimators': 770, 'min_child_weight': 3, 'subsample': 0.8599261349795126, 'colsample_bytree': 0.6082817085665551, 'gamma': 2.15200989858419, 'lambda': 0.04430542031619522, 'alpha': 0.022773160643320342, 'scale_pos_weight': 6.693397596773813}
Best CV AUC score: 0.6408

===== Detailed Cross-Validation Results with Best Parameters =====
Params: {'max_depth': 3, 'le

[I 2025-05-17 14:16:52,398] Trial 19 finished with value: 0.008640113755538414 and parameters: {'assign_payer_code': 0, 'assign_medical_specialty': 1, 'assign_weight': 0, 'assign_max_glu_serum': 0, 'assign_A1Cresult': 1}. Best is trial 11 with value: -0.04701249515674677.



Extended model (with extended)
AUC: 0.6498, Accuracy: 0.4690, F1 Score: 0.6385

Combined model (no extended)
AUC: 0.6244, Accuracy: 0.4461, F1 Score: 0.6170

Combined model (with extended)
AUC: 0.6481, Accuracy: 0.4690, F1 Score: 0.6385

Results Summary:
                            Model       AUC  Accuracy        F1  \
0        Base model (no extended)  0.614095  0.446098  0.616968   
1  Extended model (with extended)  0.649799  0.468971  0.638503   
2    Combined model (no extended)  0.624414  0.446098  0.616968   
3  Combined model (with extended)  0.648120  0.468971  0.638503   

                                                                                                                                                                                                                                                                                                                                                                                                                        

# Final Results

In [5]:
AUC Summary:
                         Model      AUC
      Base model (no extended) 0.621529
Extended model (with extended) 0.650384
  Combined model (no extended) 0.611731
Combined model (with extended) 0.627859

Base Features:
medical_specialty, weight, number_outpatient, diabetesMed, number_diagnoses, patient_nbr, admission_source_id, encounter_id, num_medications, diag_1, num_procedures, metformin, age, race, admission_type_id, time_in_hospital, insulin, diag_3, diag_2, num_lab_procedures, repaglinide, glyburide, glimepiride, glipizide, rosiglitazone, change, glyburide-metformin, acarbose, gender, pioglitazone, nateglinide, chlorpropamide, acetohexamide, tolbutamide, miglitol, citoglipton, examide, tolazamide, troglitazone, glimepiride-pioglitazone, glipizide-metformin, metformin-pioglitazone, metformin-rosiglitazone

Extended Features:
payer_code, max_glu_serum, A1Cresult, has_extended

Target - readmitted

AUC Differences:
Extended model - Combined (with extended): 0.022525000000000017
Base model - Combined (no extended):       0.009797999999999973

Standard Deviations:
Extended model - Std Dev: 0.0047
Base model - Std Dev:     0.0016


AUC Summary:
                         Model      AUC
      Base model (no extended) 0.621529
Extended model (with extended) 0.650384
  Combined model (no extended) 0.611731
Combined model (with extended) 0.627859

Base Features:
medical_specialty, weight, number_outpatient, diabetesMed, number_diagnoses, patient_nbr, admission_source_id, encounter_id, num_medications, diag_1, num_procedures, metformin, age, race, admission_type_id, time_in_hospital, insulin, diag_3, diag_2, num_lab_procedures, repaglinide, glyburide, glimepiride, glipizide, rosiglitazone, change, glyburide-metformin, acarbose, gender, pioglitazone, nateglinide, chlorpropamide, acetohexamide, tolbutamide, miglitol, citoglipton, examide, tolazamide, troglitazone, glimepiride-pioglitazone, glipizide-metformin, metformin-pioglitazone, metformin-rosiglitazone

Extended Features:
payer_code, max_glu_serum, A1Cresult, has_extended

AUC Differences:
Extended model - Combined (with extended): 0.022525000000000017
Base model - C